In [1]:
import os, sys, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

WORK = "/kaggle/working"
DATA_DIR = f"{WORK}/data"
OUT_DIR = f"{WORK}/outputs"
RESULTS_DIR = f"{WORK}/results"
for d in (DATA_DIR, OUT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

def find(name):
    hits = sorted(glob.glob(f"{WORK}/**/{name}", recursive=True)) + \
           sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found: attach the dataset/notebook that makes it (Add Input)")
    if len(hits) > 1:
        print(f"note: {len(hits)} copies of {name}, using {hits[0]}")
    return hits[0]

RAW_CSV = find("afri_med_qa_15k_v2.4_phase_2_15275.csv")

HF_USER = "Tresor26"
def hub_id(run_name):
    return f"{HF_USER}/clinicmate-{run_name}"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO = "/kaggle/temp/ClinicMate"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Tresorshingiro/ClinicMate.git {REPO}
else:
    !git -C {REPO} pull -q
sys.path.append(REPO)

import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.data import (clean_text, normalize_qtype, parse_options, parse_correct, dedup_key)

In [3]:
raw = pd.read_csv(RAW_CSV)
df = pd.DataFrame({
    "id": raw["sample_id"],
    "split": raw["split"],
    "qtype": raw["question_type"].map(normalize_qtype),
    "question": raw["question_clean"].where(raw["question_clean"].notna(), raw["question"]).map(clean_text),
    "context": raw["prompt"].map(clean_text),
    "rationale": raw["answer_rationale"].map(clean_text),
    "specialty": raw["specialty"].fillna("unknown"),
    "country": raw["country"].fillna("unknown"),
    "tier": raw["tier"].fillna("unknown"),
})
opts = raw["answer_options"].map(parse_options)
df["answer_letters"] = [parse_correct(c, o) for c, o in zip(raw["correct_answer"], opts)]
df["options"] = opts.map(lambda o: [t for _, t in o])
df["context"] = df["context"].str.replace(r"\s*Question:\s*$", "", regex=True)
print(df.shape)

(15275, 11)


In [4]:
log = {"raw": len(df)}
df = df[df["qtype"] != "unknown"];         log["known_type"] = len(df)
df = df[df["question"].str.len() > 10];    log["has_question"] = len(df)

#consumer queries have NO gold answers -> kept apart
cq = df[df["qtype"] == "cq"].copy()
cq["key"] = [dedup_key(q) for q in cq["question"]]
cq = cq.drop_duplicates("key")
log["cq_unique"] = len(cq)

#MCQ + SAQ: the answerable part
qa = df[df["qtype"].isin(["mcq", "saq"])].copy()
is_mcq = qa["qtype"] == "mcq"
valid_mcq = is_mcq & (qa["options"].str.len() >= 2) & (qa["answer_letters"].str.len() >= 1)
valid_saq = ~is_mcq & (qa["rationale"].str.split().str.len() >= 3)
qa = qa[valid_mcq | valid_saq]
log["qa_has_gold"] = len(qa)
qa["key"] = [dedup_key(q) for q in qa["question"]]
qa = qa.drop_duplicates("key")
log["qa_dedup"] = len(qa)
print(log)

{'raw': 15275, 'known_type': 15275, 'has_question': 15250, 'cq_unique': 9453, 'qa_has_gold': 5235, 'qa_dedup': 5234}


In [5]:
qa["strata"] = qa["qtype"] + "_" + qa["tier"]
train_df, tmp = train_test_split(qa, test_size=0.30, stratify=qa["strata"], random_state=42)
val_df, test_df = train_test_split(tmp, test_size=2/3, stratify=tmp["strata"], random_state=42)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, d.groupby(["qtype", "tier"]).size().to_dict())

train {('mcq', 'crowdsourced'): 90, ('mcq', 'expert'): 2722, ('saq', 'crowdsourced'): 612, ('saq', 'expert'): 239}
val {('mcq', 'crowdsourced'): 13, ('mcq', 'expert'): 388, ('saq', 'crowdsourced'): 88, ('saq', 'expert'): 34}
test {('mcq', 'crowdsourced'): 26, ('mcq', 'expert'): 779, ('saq', 'crowdsourced'): 175, ('saq', 'expert'): 68}


In [6]:
eval_df = test_df[(test_df["qtype"] == "saq") | (test_df["answer_letters"].str.len() == 1)]
print("eval:", eval_df["qtype"].value_counts().to_dict())

# official split for experiment E4
off_train, off_val = train_test_split(qa[qa["split"] == "train"], test_size=0.10, stratify=qa.loc[qa["split"] == "train", "qtype"], random_state=42)
off_test = qa[(qa["split"] == "test") & ((qa["qtype"] == "saq") | (qa["answer_letters"].str.len() == 1))]
print("official:", len(off_train), len(off_val), len(off_test))

eval: {'mcq': 739, 'saq': 243}
official: 903 101 3930


In [7]:
cols = ["id", "qtype", "question", "context", "options", "answer_letters", "rationale", "specialty", "country", "tier"]
for name, d in [("train", train_df), ("val", val_df), ("test", test_df), ("test_eval", eval_df), ("official_train", off_train), ("official_val", off_val), ("official_test", off_test), ("cq_questions", cq)]:
    d[cols].to_json(f"{DATA_DIR}/{name}.jsonl", orient="records", lines=True)
pd.Series(log).to_csv(f"{RESULTS_DIR}/data_filtering_log.csv")
print(sorted(os.listdir(DATA_DIR)))

['cq_questions.jsonl', 'official_test.jsonl', 'official_train.jsonl', 'official_val.jsonl', 'test.jsonl', 'test_eval.jsonl', 'train.jsonl', 'val.jsonl']


In [8]:
from src.prompts import build_messages, build_target
for ex in [train_df[train_df.qtype == "mcq"].iloc[0], train_df[train_df.qtype == "saq"].iloc[0]]:
    ex = ex.to_dict()
    for m in build_messages(ex): print(f"[{m['role']}]\n{m['content']}\n")
    print("[assistant]\n" + build_target(ex) + "\n" + "-" * 60)

[system]
You are ClinicMate, a medical question-answering assistant focused on healthcare in African settings. Answer accurately and concisely using established clinical knowledge and consider resource-limited settings and diseases common in Africa. You do not replace a health professional.

[user]
Answer the following multiple-choice medical question.

Question: Drug lupus-like syndrome can be due to one of the following

Options:
A. Baclofen
B. Isoniazid
C. Methotrexate
D. Rituzimab

Reply with 'Answer: <letter>' followed by a short explanation.

[assistant]
Answer: B
------------------------------------------------------------
[system]
You are ClinicMate, a medical question-answering assistant focused on healthcare in African settings. Answer accurately and concisely using established clinical knowledge and consider resource-limited settings and diseases common in Africa. You do not replace a health professional.

[user]
Question: How does the prognosis of childhood leukemia differ 